# Lab 3.1: Agentes com o Agent Service do Azure AI Foundry (20 min)

## Objetivos
- Criar um **agente standard** no Azure AI Foundry (visível no portal!)
- Entender o ciclo: **Agent → Conversation → Response**
- Usar **function calling** com agentes
- Usar o **Code Interpreter**
- Gerir o ciclo de vida do agente (criar versão, conversar, eliminar)

## Conceitos

### Agente vs Chamada ao Modelo
No Lab 3 usámos a **Responses API** — chamadas diretas ao modelo. Aqui vamos criar **agentes standard** com o **Agent Service**, visíveis no portal do Foundry:

| Responses API (Lab 3) | Agent Service (Lab 3.1) |
|---|---|
| Chamada stateless | Agente persistente no Foundry |
| Sem memória entre chamadas | `previous_response_id` ou Conversations mantêm contexto |
| Tu geres o loop de tools | O agent_reference despacha tools configurados |
| Ideal para uso simples | Ideal para assistentes complexos |

### Ciclo de Vida (Standard Agents)
```
1. Criar Agente (create_version com PromptAgentDefinition) → aparece no portal Foundry
2. Invocar via Responses API com agent_reference
3. Processar tool calls (se existirem) e submeter resultados
4. Continuar conversa com previous_response_id ou Conversations
5. Eliminar agente quando já não for necessário
```

### SDK — `azure-ai-projects` >= 2.0.0
```python
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import DefaultAzureCredential

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

# Criar agente standard
agent = project.agents.create_version(
    agent_name="meu-agente",
    definition=PromptAgentDefinition(model="gpt-4o", instructions="..."),
)

# Invocar via OpenAI Responses API
openai = project.get_openai_client()
response = openai.responses.create(
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    input="Olá!",
)
```

In [1]:
# Setup
from dotenv import load_dotenv
import os
import json

load_dotenv("../.env")

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, FunctionTool, CodeInterpreterTool
from azure.identity import DefaultAzureCredential

project_endpoint = os.getenv("AZURE_AI_AGENT_ENDPOINT")
model = os.getenv("MODEL_DEPLOYMENT", "gpt-4o")

# Cliente do projeto — para criar/gerir agentes
project = AIProjectClient(
    endpoint=project_endpoint,
    credential=DefaultAzureCredential(),
)

# Cliente OpenAI — para invocar agentes via Responses API
openai_client = project.get_openai_client()

print(f"✅ Project Client conectado: {project_endpoint}")

✅ Project Client conectado: https://foundry-mod2.services.ai.azure.com/api/projects/proj-tutor


## 🖥️ Exercício 3.1.1: Agente Simples

Vamos criar um agente standard com `PromptAgentDefinition` e invocá-lo via **Responses API** com `agent_reference`.

In [2]:
# Exercício 3.1.1: Criar um agente standard (visível no portal Foundry!)
agent = project.agents.create_version(
    agent_name="assistente-azure",
    definition=PromptAgentDefinition(
        model=model,
        instructions="És um especialista em Azure. Responde de forma concisa em português de Portugal. Usa exemplos práticos.",
    ),
)
print(f"✅ Agente criado: {agent.name} (versão {agent.version})")
print(f"   Modelo: {agent.definition.model}")
print(f"\n💡 Vai ao portal do Foundry e verifica que o agente aparece lá!")

✅ Agente criado: assistente-azure (versão 1)
   Modelo: gpt-4o

💡 Vai ao portal do Foundry e verifica que o agente aparece lá!


In [3]:
# Invocar o agente via Responses API com agent_reference
response = openai_client.responses.create(
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    input="Explica a diferença entre Azure Functions e Container Apps em 3 linhas.",
)
print(f"👤 Explica a diferença entre Azure Functions e Container Apps em 3 linhas.")
print(f"🤖 {response.output_text}")
print(f"\n   Response ID: {response.id}")

👤 Explica a diferença entre Azure Functions e Container Apps em 3 linhas.
🤖 Azure Functions é um serviço de computação serverless para executar pequenas funções de forma escalável, ideal para tarefas event-driven (e.g., processar uma mensagem de fila). Azure Container Apps permite implementar aplicações completas em contentores, oferecendo maior controlo sobre configurações e ambientes personalizados. Functions é melhor para lógica pequena, enquanto Container Apps suporta aplicações mais complexas.

   Response ID: resp_005c1fc0dbc9c7d50069dd34947610819493ac19464961a5da


In [4]:
# Continuar a conversa usando previous_response_id (o agente mantém contexto!)
follow_up = openai_client.responses.create(
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    previous_response_id=response.id,
    input="E qual dos dois recomendas para um webhook simples?",
)
print(f"👤 E qual dos dois recomendas para um webhook simples?")
print(f"🤖 {follow_up.output_text}")

👤 E qual dos dois recomendas para um webhook simples?
🤖 Para um webhook simples, recomendo **Azure Functions**, pois é otimizado para tarefas leves e event-driven, como responder rapidamente a chamadas HTTP. Além disso, reduz custos e complexidade, já que apenas pagas pelo tempo de execução.


## 🖥️ Exercício 3.1.2: Agente com Function Calling

O verdadeiro poder dos agentes é usar **ferramentas**. Vamos criar um agente com `FunctionTool` que pode consultar informações sobre preços e regiões Azure. O ciclo é: invocar → processar tool calls → submeter resultados → obter resposta final.

In [5]:
# Exercício 3.1.2: Definir funções e criar agente com tools
from openai.types.responses.response_input_param import FunctionCallOutput

# Funções que o agente pode chamar
def obter_preco_servico(servico: str) -> str:
    """Retorna o preço estimado de um serviço Azure."""
    precos = {
        "app service basic": "€50/mês",
        "container apps": "€0.000012/vCPU-s",
        "functions consumption": "€0.000016/GB-s",
        "aks": "Grátis (control plane) + custo dos nodes",
        "cosmos db": "A partir de €25/mês (serverless)",
    }
    return json.dumps({"servico": servico, "preco": precos.get(servico.lower(), "Preço não disponível. Consulta azure.com/pricing")})

def obter_regiao_recomendada(caso_uso: str) -> str:
    """Recomenda a melhor região Azure para um caso de uso."""
    regioes = {
        "europa": "West Europe (Holanda) ou North Europe (Irlanda)",
        "portugal": "West Europe (mais próximo de Portugal)",
        "global": "Usa Traffic Manager com múltiplas regiões",
        "ai": "East US ou Sweden Central (melhor disponibilidade de modelos)",
    }
    return json.dumps({"caso_uso": caso_uso, "recomendacao": regioes.get(caso_uso.lower(), regioes["europa"])})

# Mapa de funções para despacho
func_map = {
    "obter_preco_servico": obter_preco_servico,
    "obter_regiao_recomendada": obter_regiao_recomendada,
}

# Definir tools com schemas explícitos (FunctionTool do azure-ai-projects)
tools = [
    FunctionTool(
        name="obter_preco_servico",
        description="Retorna o preço estimado de um serviço Azure.",
        parameters={
            "type": "object",
            "properties": {"servico": {"type": "string", "description": "Nome do serviço Azure"}},
            "required": ["servico"],
            "additionalProperties": False,
        },
        strict=True,
    ),
    FunctionTool(
        name="obter_regiao_recomendada",
        description="Recomenda a melhor região Azure para um caso de uso.",
        parameters={
            "type": "object",
            "properties": {"caso_uso": {"type": "string", "description": "O caso de uso (ex: portugal, europa, ai)"}},
            "required": ["caso_uso"],
            "additionalProperties": False,
        },
        strict=True,
    ),
]

# Criar agente com tools
agent_tools = project.agents.create_version(
    agent_name="consultor-azure",
    definition=PromptAgentDefinition(
        model=model,
        instructions="És um consultor Azure. Usa as ferramentas disponíveis para dar informações precisas sobre preços e regiões. Responde em português.",
        tools=tools,
    ),
)
print(f"✅ Agente com tools criado: {agent_tools.name} (versão {agent_tools.version})")

✅ Agente com tools criado: consultor-azure (versão 1)


In [6]:
# Testar o agente com function calling
response_tools = openai_client.responses.create(
    extra_body={"agent_reference": {"name": agent_tools.name, "type": "agent_reference"}},
    input="Quero criar uma app com Container Apps. Quanto custa e qual a melhor região para Portugal?",
)

# Processar tool calls e submeter resultados
tool_outputs = []
for item in response_tools.output:
    if item.type == "function_call":
        print(f"[Tool] {item.name}({item.arguments})")
        result = func_map[item.name](**json.loads(item.arguments))
        tool_outputs.append(
            FunctionCallOutput(type="function_call_output", call_id=item.call_id, output=result)
        )

# Submeter resultados e obter resposta final
if tool_outputs:
    response_tools = openai_client.responses.create(
        input=tool_outputs,
        previous_response_id=response_tools.id,
        extra_body={"agent_reference": {"name": agent_tools.name, "type": "agent_reference"}},
    )

print(f"\n👤 Quero criar uma app com Container Apps. Quanto custa e qual a melhor região para Portugal?")
print(f"🤖 {response_tools.output_text}")

[Tool] obter_preco_servico({"servico":"Container Apps"})
[Tool] obter_regiao_recomendada({"caso_uso":"portugal"})

👤 Quero criar uma app com Container Apps. Quanto custa e qual a melhor região para Portugal?
🤖 Os preços para o serviço **Container Apps** no Azure começam em **€0.000012 por segundo de CPU (vCPU-s)**. O custo final dependerá do número de instâncias, tempo de execução e outros componentes que utilizar na sua aplicação.

A região recomendada para Portugal é **West Europe**, pois é a mais próxima e oferece baixa latência.


## 🖥️ Exercício 3.1.3: Code Interpreter

O **Code Interpreter** permite ao agente escrever e executar código Python autonomamente. Basta adicioná-lo como tool na definição do agente.

In [7]:
# Exercício 3.1.3: Agente com Code Interpreter
agent_code = project.agents.create_version(
    agent_name="analista-dados",
    definition=PromptAgentDefinition(
        model=model,
        instructions="És um analista de dados. Usa o code interpreter para fazer cálculos e criar visualizações. Responde em português.",
        tools=[CodeInterpreterTool()],
    ),
)
print(f"✅ Agente com code interpreter: {agent_code.name} (versão {agent_code.version})")

response_code = openai_client.responses.create(
    extra_body={"agent_reference": {"name": agent_code.name, "type": "agent_reference"}},
    input="Calcula a sequência de Fibonacci até ao 10º número e mostra o resultado numa tabela formatada.",
)
print(f"\n🤖 {response_code.output_text}")

✅ Agente com code interpreter: analista-dados (versão 1)

🤖 Aqui está a tabela com os números da sequência de Fibonacci até ao 10º número:

| Posição (n) | Número de Fibonacci |
|-------------|---------------------|
| 1           | 0                   |
| 2           | 1                   |
| 3           | 1                   |
| 4           | 2                   |
| 5           | 3                   |
| 6           | 5                   |
| 7           | 8                   |
| 8           | 13                  |
| 9           | 21                  |
| 10          | 34                  |


## 🧹 Limpeza

Os agentes são **persistentes** — é importante eliminá-los quando já não são necessários.

In [8]:
# Limpeza - eliminar todos os agentes criados
for name in ["assistente-azure", "consultor-azure", "analista-dados"]:
    try:
        project.agents.delete(agent_name=name)
        print(f"🗑️ Agente '{name}' eliminado")
    except Exception as e:
        print(f"⚠️ Erro ao eliminar '{name}': {e}")

print("\n✅ Limpeza concluída! Verifica no portal do Foundry que os agentes desapareceram.")

🗑️ Agente 'assistente-azure' eliminado
🗑️ Agente 'consultor-azure' eliminado
🗑️ Agente 'analista-dados' eliminado

✅ Limpeza concluída! Verifica no portal do Foundry que os agentes desapareceram.


## ✅ Resumo

Neste lab aprendeste:
- Criar **agentes standard** com `PromptAgentDefinition` via `azure-ai-projects` >= 2.0.0
- Invocar agentes via **Responses API** com `agent_reference`
- Manter **contexto** entre turnos usando `previous_response_id`
- Usar **function calling** com o ciclo: invocar → processar tool calls → submeter resultados
- Usar o **Code Interpreter** para computação autónoma
- Gerir o ciclo de vida (criar versão e eliminar agentes)

### Lab 3 vs Lab 3.1
| Lab 3 (Responses API) | Lab 3.1 (Agent Service) |
|---|---|
| Chamadas diretas ao modelo | Agentes standard no portal Foundry |
| Tu geres o contexto | `previous_response_id` ou Conversations |
| Tools definidos inline | Tools associados ao agente (persistentes) |
| Mais simples e leve | Mais poderoso para assistentes complexos |

**Próximo:** [Lab 4 - Knowledge e RAG](lab04-knowledge-rag.ipynb)